# SPECIATE v5.4 cleanup

Cleans the US EPA [SPECIATE](https://www.epa.gov/air-emissions-modeling/speciate) v5.4 database (speciation profiles of organic gases and particulate matter from emission sources) so that it contains individual chemicals with standardized identifiers.

**Workflow**

| Step | What happens |
| --- | --- |
| 0 | Build a flat table from the Access (`.accdb`) database |
| 1 | Clean names and CAS values, remove mixtures |
| 2 | Add `weight_fraction` |
| 3 | Format CAS numbers (`cas`) |
| 4 | Retrieve compound properties (InChIKey, title, IUPAC name) from PubChem; resumable, cached |
| 5 | Optionally save the cleaned intermediate table |
| 6 | Merge the PubChem data into the table |
| 7 | Remove entries without InChIKey, by InChIKey list and by keyword; sort and save |
| 8-9 | Descriptive statistics and export per compound |

All paths and settings are in the configuration cell at the top (input database, output files, cache, PubChem rate limits); adjust them first. Run the notebook cell by cell. Step 4 needs internet access and, for the full table, takes a while (several thousand PubChem requests); it can be interrupted and re-run, and finished lookups are kept in the cache file.

In [1]:
# Reset env to erase previous vars, dicts, dfs, etc.
%reset -f

from datetime import datetime

print(f"Environment reset | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Environment reset | 2026-11-18 12:36:10



In [2]:
# Configuration & imports
import os
import re
import time
import threading
from collections import defaultdict, OrderedDict
from datetime import datetime
from urllib.parse import quote

import chardet
import numpy as np
import pandas as pd
import requests
from access_parser import AccessParser  # pure-Python .accdb reader (pip install access-parser)

# Shared helpers/filters (common lives in the parent folder of this notebook)
import sys
sys.path.insert(0, os.path.abspath(".."))
from common.utils import print_overwrite, df_stats
from common.compound_filters import apply_standard_filters
from common.pubchem import PubChemClient, lookup_attempts, per_row, run_cached_lookup
from common.cleaning import add_uid, clean_names, format_cas, read_cache

# ---------------------------------------------------------------------------
# CONFIGURATION (all user-editable settings live here)
# ---------------------------------------------------------------------------
# Input: SPECIATE Access database (downloaded from the US EPA)
ACCDB_PATH = "../original_databases/final_speciate_5.4_2025-09-08/SPECIATE 5.4 2025-09-08.accdb"

# Output & cache folders (subfolders of the notebook folder)
BASE_DIR = "."
OUTPUT_DIR = os.path.join(BASE_DIR, "speciate_cleaned")
CACHE_DIR = os.path.join(BASE_DIR, "cache")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Cache file (resumable PubChem lookups, one row per unique identifier set)
PUBCHEM_CACHE_CSV = os.path.join(CACHE_DIR, "PubChem_lookup_cache.csv")

# Output files
CSV_PATH = os.path.join(OUTPUT_DIR, "SPECIATE_Profiles_All.csv")                 # flat table built in step 0; input for step 1
INTERMEDIATE_CSV = os.path.join(OUTPUT_DIR, "SPECIATE_Profiles_All_01_Fltrd.csv")   # cleaned table before InChIKey/keyword filtering (step 5)
FLTRD_CSV = os.path.join(OUTPUT_DIR, "SPECIATE_Profiles_All_02_Fltrd.csv")          # final filtered table (step 7b)
STATS_CSV = os.path.join(OUTPUT_DIR, "SPECIATE_Profiles_All_03_CAS_Profile_Composition.csv")  # per-compound statistics (step 9)

# PubChem enrichment settings
LOOKUP_LIMIT = None      # set to e.g. 50 for a quick test run; None = all compounds
MAX_REQ_PER_SEC = 4      # PubChem asks for <= 5 requests/second. You may be blocked when increasing the rate.
N_WORKERS = 4
CHUNK = 200              # compounds per cache write; smaller chunks give a more responsive ETA

print(f"Configuration & imports done | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Configuration & imports done | 2026-11-18 12:36:10



In [3]:
# Helper to remove previous progress message
# print_overwrite is imported from inquire_common.utils in the configuration cell

print(f"Helper to remove previous progress message created | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


Helper to remove previous progress message created | 2026-11-17 16:17:04



## Step 0: Build the flat SPECIATE table from the Access database

The cleanup below expects a single flat table with one row per profile-species pair. The EPA distributes SPECIATE 5.4 as a relational Microsoft Access database (`.accdb`), so the next two cells read it directly and build that table. The cleanup notebook originally started from a ready-made CSV (`SPECIATE_Profiles_All_02.csv`) whose origin is not documented; the table built here is meant to replicate its columns.

Tables used:

| Table | Role | Join key |
| --- | --- | --- |
| `SPECIES` | one row per species in a profile, with `WEIGHT_PERCENT` (the base table) | |
| `PROFILES` | profile metadata (`PROFILE_NAME`, categories, keywords) | `PROFILE_CODE` |
| `SPECIES_PROPERTIES` | species identity (`CAS`, `CAS no hyphen`, `DSSTox_ID`, `SPECIES_NAME`, SMILES) | `SPECIES_ID` |
| `tblSpeciesAndConcatSynonyms` | synonyms joined into one ` \|\| `-separated string (`SPECIES_NAMES`) | `SPECIES_ID` |

`SPECIES_PROPERTIES.SPECIES_NAME` is renamed to `orig_name`. This mapping is an assumption: the notebook never states where `orig_name` comes from, but its cleaning rules (synonym and mixture patterns) fit a species name.

Set `ACCDB_PATH` in the configuration cell to the location of the downloaded database. Parsing with `access-parser` is pure Python and may take a minute or two for the 348k-row `SPECIES` table.

In [4]:
# Check the input Access database (only needed if the flat CSV does not exist yet)
if os.path.exists(CSV_PATH):
    db = None
    print(f"{CSV_PATH} already exists; skipping Access database read.\n")
else:
    assert os.path.exists(ACCDB_PATH), f"Access database not found: {ACCDB_PATH}"

    # Open the database and list the user tables (skip Access system tables, which start with 'MSys')
    db = AccessParser(ACCDB_PATH)
    print("Tables in database:")
    for table_name in db.catalog:
        if not table_name.startswith("MSys"):
            print(" ", table_name)
    print()


./speciate_cleaned/SPECIATE_Profiles_All.csv already exists; skipping Access database read.



In [5]:
# Join SPECIATE tables into one flat table and write it to CSV (only if CSV_PATH does not exist)
if os.path.exists(CSV_PATH):
    print(f"{CSV_PATH} already exists; skipping build from Access database.\n")
else:
    print("Building flat SPECIATE table from Access database.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
    print("It will take a while...", "\n")

    # Parse the four tables needed into DataFrames
    tabs = {
        t: pd.DataFrame(db.parse_table(t))
        for t in ["SPECIES", "PROFILES", "SPECIES_PROPERTIES", "tblSpeciesAndConcatSynonyms"]
    }

    # Base table: one row per species within a profile (contains WEIGHT_PERCENT)
    species = tabs["SPECIES"]

    # Keep only the metadata columns that are needed downstream
    profiles = tabs["PROFILES"][[
        "PROFILE_CODE", "PROFILE_NAME", "PROFILE_TYPE", "MASTER_POLLUTANT", "VERSION",
        "CATEGORY_LEVEL_1_Generation_Mechanism", "CATEGORY_LEVEL_2_Sector_Equipment",
        "CATEGORY_LEVEL_3_ Fuel_Product", "Keywords",
    ]]
    props = tabs["SPECIES_PROPERTIES"][[
        "SPECIES_ID", "CAS", "CAS no hyphen", "DSSTox_ID", "SPECIES_NAME", "Molecular Formula", "Smiles Notation",
    ]]
    syn = tabs["tblSpeciesAndConcatSynonyms"][["SPECIES_ID", "SPECIES_NAMES"]]

    # The lookup tables must have unique keys, otherwise the joins would duplicate rows
    for label, d, key in [("PROFILES", profiles, "PROFILE_CODE"),
                          ("SPECIES_PROPERTIES", props, "SPECIES_ID"),
                          ("synonyms", syn, "SPECIES_ID")]:
        assert not d[key].duplicated().any(), f"Duplicate {key} in {label}"

    # Left joins keep every SPECIES row; validate='m:1' raises if a lookup key is not unique
    flat = (
        species
        .merge(profiles, on="PROFILE_CODE", how="left", validate="m:1")
        .merge(props, on="SPECIES_ID", how="left", validate="m:1")
        .merge(syn, on="SPECIES_ID", how="left", validate="m:1")
        .rename(columns={"SPECIES_NAME": "orig_name"})  # assumption: 'orig_name' in the cleanup = species name
    )
    assert len(flat) == len(species), "Row count changed during joins"

    # Atomic write: write to a temp file in the same folder, then rename onto CSV_PATH
    tmp_path = CSV_PATH + ".tmp"
    try:
        flat.to_csv(tmp_path, index=False, encoding="utf-8")
        os.replace(tmp_path, CSV_PATH)
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

    print(f"Flat SPECIATE table written to {CSV_PATH}.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
    print(f"\nShape: \n{flat.shape[0]:,} rows × {flat.shape[1]} columns\n")


./speciate_cleaned/SPECIATE_Profiles_All.csv already exists; skipping build from Access database.



### Step 0b: Locate the flat CSV

Confirm that the CSV from step 0 exists. To start from a different flat SPECIATE CSV instead (e.g. an EPA download with the same columns), skip step 0 and set `CSV_PATH` in the configuration cell.

In [6]:
# Check that the flat SPECIATE CSV written in step 0 exists
# (CSV_PATH is set in the configuration cell; to use a different CSV, e.g. an EPA download, change it there)
assert os.path.exists(CSV_PATH), f"CSV file not found: {CSV_PATH}"
print("CSV path is valid.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


CSV path is valid. 2026-11-17 16:17:04 



Read the CSV into `df`. The encoding is first guessed with `chardet` from the first 100 kB.

In [7]:
# Read CSV file
# Check encoding
with open(CSV_PATH, "rb") as f:
    result = chardet.detect(f.read(100000))
print(f"CSV file encoding: \n{result}\n")

print("Reading CSV with pandas.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

# Read entire CSV
df = pd.read_csv(CSV_PATH, encoding=result["encoding"], low_memory=False)

# Standard schema: a flat table built by an earlier version of this notebook still has the old 'Name' header
df = df.rename(columns={"Name": "orig_name"})

print("CSV loaded.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print(f"\nCSV shape: \n{df.shape[0]:,} rows × {df.shape[1]} columns\n")


CSV file encoding: 
{'encoding': 'utf-8', 'confidence': 0.8000456, 'language': 'en', 'mime_type': 'text/plain'}

Reading CSV with pandas. 2026-11-17 16:17:04
CSV loaded. 2026-11-17 16:17:05

CSV shape: 
348,268 rows × 25 columns



### Step 1: Clean names and CAS values and remove mixtures

This cell works on a copy (`df_rename`) and keeps the originals in `orig_name_Original` and `SPECIES_NAMES_Original`. In order:

1. **Odd characters:** removes runs of `»` from `CAS`, `orig_name`, `CAS no hyphen` and `SPECIES_NAMES`, and strips whitespace.
2. **Synonyms:** in `orig_name` and `SPECIES_NAMES`, cuts everything from ` (or` or ` ||` onwards, keeping only the first name. In SPECIATE, ` || ` separates synonyms.
3. **Mixtures by name:** drops rows whose `orig_name` contains `;`, `/`, `+` or `&`.
4. **Mixtures by CAS:** drops rows whose `CAS` contains the same separators.

Each stage prints counts of rows affected. The cleaned table is `df_clean_cas`.

In [8]:
# Check Name & CAS for odd values and edit the name
print("Checking 'orig_name' column for odd values.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Keep original names for diagnostics
df["orig_name_Original"] = df["orig_name"]
df["SPECIES_NAMES_Original"] = df["SPECIES_NAMES"]

# Remove '»', keep the first synonym of each name, then drop mixtures (names with ; / + & and CAS with the same)
df_clean_cas = clean_names(
    df,
    name_cols=["orig_name", "SPECIES_NAMES"],
    cas_cols=["CAS", "CAS no hyphen"],
    separators=[";", "/", "+", "&"],
)


Checking 'Name' column for odd values. 2026-11-17 16:17:05 

Column 'CAS': '»' removed from 0 rows
Column 'CAS no hyphen': '»' removed from 0 rows
Column 'Name': '»' removed from 0 rows
Column 'SPECIES_NAMES': '»' removed from 0 rows

Total 'Name' values: 
348,268

'Name' containing synonym patterns (to be fixed): 
58,288

'Name' actually fixed: 
58,288

Total 'SPECIES_NAMES' values: 
348,268

'SPECIES_NAMES' containing synonym patterns (to be fixed): 
286,932

'SPECIES_NAMES' actually fixed: 
289,600

Total rows before Names filtration: 
348,268

Names containing mixture patterns (to be removed): 
9,676

Names actually removed: 
9,676

Remaining rows after filtering: 
338,592

Number of Names containing each separator pattern:
  ';': 5,296
  '/': 1,218
  '+': 669
  '&': 2,738

Total rows before CAS# filtration: 
338,592

CAS# containing mixture patterns (to be removed): 
35

CAS# actually removed: 
35

Remaining rows after filtering: 
338,557

Number of CAS# containing each separator 

### Step 2: Add `weight_fraction`

`weight_fraction = WEIGHT_PERCENT / 100`, inserted just before `WEIGHT_PERCENT`. `orig_name_Original` and `SPECIES_NAMES_Original` (the untouched names kept in step 1) are moved next to the cleaned columns. Rows with a missing `WEIGHT_PERCENT` get a missing `weight_fraction`.

In [9]:
# Create weight_fraction column
print("Creating 'weight_fraction' column.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

df_move = df_clean_cas.copy()

wt_pc = df_move["WEIGHT_PERCENT"]

df_move["weight_fraction"] = (wt_pc / 100)

cols = list(df_move.columns)

cols.remove("weight_fraction")
wt_pc_idx = cols.index("WEIGHT_PERCENT")
cols.insert(wt_pc_idx, "weight_fraction")

cols.remove("orig_name_Original")
cols.insert(wt_pc_idx - 1, "orig_name_Original")

cols.remove("SPECIES_NAMES_Original")
cols.insert(wt_pc_idx + 6, "SPECIES_NAMES_Original")

df_wf = df_move[cols].copy()

print("Created & populated 'weight_fraction' column.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


Creating 'weight_fraction' column. 2026-11-17 16:17:08 

Created & populated 'weight_fraction' column. 2026-11-17 16:17:09 



### Step 3: Format CAS numbers

`format_cas` rebuilds a standard hyphenated CAS number (`XXXXXXX-YY-Z`) from the un-hyphenated `CAS no hyphen` column: it strips whitespace and `»`, drops any decimal part, returns `NaN` if the value is missing or fewer than 4 characters remain, and re-inserts hyphens before the last two digits and before the check digit. The result is stored in the new `cas` column, placed right after `CAS`.

Note: missing values stay `NaN` (a real null). The text conversion (`.astype(str)`, which turns `NaN` into `"nan"`) is applied later, when hashing for the PubChem cache.

In [10]:
# Format CAS (missing values stay NaN)
print("Formatting CAS.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# format_cas is imported from inquire_common.cleaning
df_wf["cas"] = df_wf["CAS no hyphen"].apply(format_cas)

cols_df_wf = list(df_wf.columns)
cas_idx = cols_df_wf.index("CAS")
cols_df_wf.remove("cas")
cols_df_wf.insert(cas_idx + 1, "cas")
df_wf = df_wf[cols_df_wf].copy()

print("Done formatting CAS#.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")
print(f"New CAS# column: \n{df_wf['cas']}\n")

Formatting CAS. 2026-11-17 16:17:09 

Done formatting CAS#. 2026-11-17 16:17:09 

New CAS# column: 
0            71-55-6
1            79-00-5
2          3073-66-3
3            95-63-6
4           108-67-8
             ...    
338552       74-98-6
338553      108-88-3
338554           NaN
338555    73513-42-5
338556           NaN
Name: CAS#, Length: 338557, dtype: object



### Step 4: Retrieve compound properties from PubChem

This step replaces the external PubChem_Retriever / Data_Retriever tools and works as in the FooDB notebook: requests go to the PubChem PUG REST API, are rate-limited (`MAX_REQ_PER_SEC`, with extra delay if PubChem's `X-Throttling-Control` header reports load), and results are written to a cache file as they come in, so the step can be interrupted and re-run.

Each distinct combination of `orig_name`, `SPECIES_NAMES`, `cas`, `DSSTox_ID` and `Smiles Notation` gets a hash-based `uid`. For each one the lookup tries, in order, until PubChem returns a hit:

1. `cas` (PubChem resolves CAS numbers as synonyms)
2. `Smiles Notation`
3. `orig_name`
4. `SPECIES_NAMES`

The method that succeeded is stored in `Lookup_method`. Compounds PubChem cannot resolve are cached as `not_found`, so they are not retried on re-runs (delete their rows from the cache to retry). Compounds that fail because of a transient error are not cached and are retried on the next run.

Caveats: the first hit is taken when a query matches several PubChem compounds, and a CAS or name lookup can match a different compound than intended. Set `LOOKUP_LIMIT` in the configuration cell for a quick test run.

The first cell defines the helper functions; the second runs the lookup.

In [11]:
# PubChem client (rate limiter and request helpers live in inquire_common/pubchem.py)
client = PubChemClient(max_req_per_sec=MAX_REQ_PER_SEC)

def lookup_compound(row):
    """row: (uid, cas, smiles, name, species_name). Tries CAS#, SMILES, Name, SPECIES_NAMES in order."""
    uid, cas, smiles, name, species_name = row
    return lookup_attempts(uid, [
        ("CAS", cas, client.name_lookup_row),
        ("SMILES", smiles, client.smiles_lookup_row),
        ("Name", name, client.name_lookup_row),
        ("SPECIES_NAMES", species_name, client.name_lookup_row),
    ])

print(f"PubChem helpers defined | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


PubChem helpers defined | 2026-11-17 16:17:09



In [12]:
# Retrieve PubChem data for the unique identifier sets (resumable; results are appended to the cache)
print("Creating unique 'orig_name' & 'SPECIES_NAMES' & 'cas' & 'DSSTox_ID' & 'Smiles Notation' set for PubChem data retrieval.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Distinct identifier sets; the uid is a hash of the values, so the cache stays valid if row order changes
UNIQUE_KEY_COLS = ["orig_name", "SPECIES_NAMES", "cas", "DSSTox_ID", "Smiles Notation"]

# Text form of CAS# (missing -> 'nan') in both df_wf and df_unique, so the uid hash is stable and the
# step 6 merge on the key columns matches.
df_wf["cas"] = df_wf["cas"].astype(str)

df_unique = add_uid(df_wf, UNIQUE_KEY_COLS)
print(f"Unique identifier sets: {len(df_unique):,}\n")

cache_cols = ["uid", "CID", "Name_PubChem", "IUPACName_PubChem", "InChIKey_PubChem", "Lookup_method"]
cache_current = run_cached_lookup(
    df_unique, ["cas", "Smiles Notation", "orig_name", "SPECIES_NAMES"], per_row(lookup_compound, N_WORKERS),
    PUBCHEM_CACHE_CSV, cache_cols, client, chunk=CHUNK, limit=LOOKUP_LIMIT,
)


Creating unique 'Name' & 'SPECIES_NAMES' & 'CAS#' & 'DSSTox_ID' & 'Smiles Notation' set for PubChem data retrieval. 2026-11-17 16:17:10 

Unique identifier sets: 2,870

2,870 of 2,870 compounds already cached, 0 to look up | 2026-11-17 16:17:10
No up-front time estimate: an Estimated Time of Arrival based on measured throughput is shown while running.
It is safe to interrupt and re-run.



Cached compounds: 2,870 of 2,870
Resolved to an InChIKey: 2,869
Lookup method:
Lookup_method
CAS          2035
SMILES        833
Name            1
not_found       1

PubChem retrieval done | 2026-11-17 16:17:10



### Step 5: Save the cleaned intermediate table (optional)

Writes the full cleaned table (`df_wf`) to CSV (`INTERMEDIATE_CSV` in the configuration cell), before InChIKey/keyword filtering.

In [13]:
# Write cleaned SPECIATE to CSV file (path set in the configuration cell)
print("Writing filtered SPECIATE to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Prevent Excel CAS# conversion
df_wf["cas"] = df_wf["cas"].astype(str)
# Save SPECIATE BEFORE further filtration
df_wf.to_csv(INTERMEDIATE_CSV, index=False)

print(f"Filtered SPECIATE written to {INTERMEDIATE_CSV}.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

print(f"Filtered DF shape: \n{df_wf.shape[0]:,} rows × {df_wf.shape[1]} columns\n")


Writing filtered SPECIATE to CSV. 2026-11-17 16:17:10 

Filtered SPECIATE written to ./speciate_cleaned/SPECIATE_Profiles_All_01_Fltrd.csv. 2026-11-17 16:17:13 

Filtered DF shape: 
338,557 rows × 29 columns



QC print of the columns in the intermediate table.

In [14]:
# QC print
print(f"Filtered SPECIATE (intermediate) columns:")
for col in df_wf.columns:
    print(col)
print()


Filtered SPECIATE (intermediate) columns:
Name_Original
SPECIES_ID
weight_fraction
WEIGHT_PERCENT
UNCERTAINTY_PERCENT
SPECIES_EMISSION_RATE
PROFILE_CODE
SPECIES_NAMES_Original
INCLUDE_IN_SUM
UNCERTAINTY_METHOD
ANALYTICAL_METHOD
PHASE
SPECIES_EMISSION_RATE_UNIT
PROFILE_NAME
PROFILE_TYPE
MASTER_POLLUTANT
VERSION
CATEGORY_LEVEL_1_Generation_Mechanism
CATEGORY_LEVEL_2_Sector_Equipment
CATEGORY_LEVEL_3_ Fuel_Product
Keywords
CAS
CAS#
CAS no hyphen
DSSTox_ID
Name
Molecular Formula
Smiles Notation
SPECIES_NAMES



### Step 6: Merge the PubChem data into the SPECIATE table

Joins the cache from step 4 onto the full table (`df_wf`) using the five identifier columns, giving `df_xl`. Three PubChem columns are added, named so that the filtering below works unchanged:

| Column | Content |
| --- | --- |
| `name` | PubChem title of the matched compound |
| `inchikey` | PubChem InChIKey |
| `IUPACName` | PubChem IUPAC name |

`CID`, the PubChem compound ID, and `Lookup_method` (`CAS`, `SMILES`, `Name`, `SPECIES_NAMES` or `not_found`) are added too. `validate="m:1"` makes the merge fail if the lookup table were not unique per identifier set, and the row count is asserted to be unchanged. Compounds that PubChem could not resolve keep missing values and are removed in step 7a.

In [15]:
# Merge the PubChem results (cache from step 4) back onto the full table
print("Merging PubChem data into SPECIATE.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

pubchem = (
    read_cache(PUBCHEM_CACHE_CSV)
    .rename(columns={
        "Name_PubChem": "name",
        "IUPACName_PubChem": "IUPACName",
        "InChIKey_PubChem": "inchikey",
    })
)

# One row per unique identifier set, now with the PubChem columns
df_unique_pc = df_unique.merge(pubchem, on="uid", how="left", validate="1:1")
n_missing_lookup = df_unique_pc["Lookup_method"].isna().sum()
if n_missing_lookup:
    print(f"WARNING: {n_missing_lookup:,} unique compounds have no cached result yet. Re-run step 4 to retrieve them.\n")

# Many-to-one join back onto the full table (rows with missing key values match each other in pandas merges)
df_xl = df_wf.merge(
    df_unique_pc.drop(columns="uid"),
    on=UNIQUE_KEY_COLS,
    how="left",
    validate="m:1",
)
assert len(df_xl) == len(df_wf), "Row count changed during the PubChem merge"

print(f"Rows with InChIKey: {df_xl['inchikey'].notna().sum():,} of {len(df_xl):,}")
print(f"\nSPECIATE + PubChem shape: \n{df_xl.shape[0]:,} rows × {df_xl.shape[1]} columns\n")
print("Done merging PubChem data.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


Merging PubChem data into SPECIATE. 2026-11-17 16:17:14 

Rows with InChIKey: 338,554 of 338,557

SPECIATE + PubChem shape: 
338,557 rows × 34 columns

Done merging PubChem data. 2026-11-17 16:17:14 



QC print of the columns in the retrieved-data table. The filtering step below requires `inchikey` and `name` among them.

In [16]:
# QC print
print(f"Updated SPECIATE columns:")
for col in df_xl.columns:
    print(col)
print()


Updated SPECIATE columns:
Name_Original
SPECIES_ID
weight_fraction
WEIGHT_PERCENT
UNCERTAINTY_PERCENT
SPECIES_EMISSION_RATE
PROFILE_CODE
SPECIES_NAMES_Original
INCLUDE_IN_SUM
UNCERTAINTY_METHOD
ANALYTICAL_METHOD
PHASE
SPECIES_EMISSION_RATE_UNIT
PROFILE_NAME
PROFILE_TYPE
MASTER_POLLUTANT
VERSION
CATEGORY_LEVEL_1_Generation_Mechanism
CATEGORY_LEVEL_2_Sector_Equipment
CATEGORY_LEVEL_3_ Fuel_Product
Keywords
CAS
CAS#
CAS no hyphen
DSSTox_ID
Name
Molecular Formula
Smiles Notation
SPECIES_NAMES
CID
Feature name
IUPACName
InChIKey
Lookup_method



### Step 7a: Remove non-individual-chemical entries

Filters the table in three stages, each printing what was dropped:

1. **No InChIKey:** rows without an InChIKey are removed. These are mixtures or unidentified features that PubChem could not resolve to a single structure.
2. **Specific InChIKeys:** rows whose InChIKey is in `drop_inchikeys` are removed (elements, inorganic ions and salts, acids, isotopically labelled standards, polymers and similar). The trailing comments in the list give the compound each key stands for.
3. **Keywords in `name`:** rows whose name matches `unwanted_pattern` (a `|`-separated list of metals, minerals, inorganic compounds, mixtures and biological materials) are removed. Single-word terms get `\b` word boundaries so that, for example, `lead` does not match `leading`; multi-word terms are matched as-is. The match is case-insensitive.

The keyword and InChIKey lists are curated by hand, so review them against your own scope before relying on the output.

In [17]:
# Filter database by InChIKey & keywords
print("Filtering SPECIATE by InChIKey & other keywords.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Drops rows without InChIKey (mixtures or unidentified features), listed InChIKeys (mixtures, inorganic compounds,
# etc.) and unwanted name keywords. Lists live in inquire_common/filters: drop_inchikeys.txt,
# keywords_common.txt + keywords_speciate.txt, keyword_exceptions.txt.
# anchor_end=True keeps whole-word matching; False also matches stems (magnes, mercur, ...) and removes more entries.
initial_n = len(df_xl)
df_inchikey, dropped_df, dropped_chems = apply_standard_filters(
    df_xl, source="speciate", anchor_end=True, label="SPECIATE",
)

print("Done keyword-filtering SPECIATE.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


Filtering SPECIATE by InChIKey & other keywords. 2026-11-17 16:17:14 

Number of entries in SPECIATE before InChIKey & keyword filtration: 
338,557

Dropped 3 entries with no InChIKey values. 

Remaining rows: 
338,554

Dropped compounds by InChIKey:
AKEJUJNQAAGONA-UHFFFAOYSA-N >>> Sulfur trioxide
ATJFFYVFTNAWJD-UHFFFAOYSA-N >>> Tin atom
BVKZGUZCCUSVTD-UHFFFAOYSA-L >>> Carbonate Ion
GDTBXPJZTBHREO-UHFFFAOYSA-N >>> Bromine
HEDRZPFGACZZDS-UHFFFAOYSA-N >>> Chloroform
KEAYESYHFKHZAL-UHFFFAOYSA-N >>> Sodium
NBIIXXVUZAFLBC-UHFFFAOYSA-K >>> Phosphate
NHNBFGGVMKEFGY-UHFFFAOYSA-N >>> Nitrate
NINIDFKCEFEMDL-UHFFFAOYSA-N >>> Sulfur
OAICVXFJPJFONN-UHFFFAOYSA-N >>> Phosphorus
OKTJSMMVPCPJKN-UHFFFAOYSA-N >>> Carbon
PXGOKWXKJXAPGV-UHFFFAOYSA-N >>> Fluorine
QAOWNCQODCNURD-UHFFFAOYSA-L >>> Sulfate Ion
QGZKDVFQNNGYKY-UHFFFAOYSA-N >>> Ammonia
QGZKDVFQNNGYKY-UHFFFAOYSA-O >>> Ammonium
QMMFVYPAHWMCMS-UHFFFAOYSA-N >>> Dimethyl Sulfide
QTBSBXVTEAMEQO-UHFFFAOYSA-M >>> Acetate
RAHZWNYVWXNFOC-UHFFFAOYSA-N >>> Su

### Step 7b: Sort and save the filtered SPECIATE

Sorts the filtered table by `CATEGORY_LEVEL_1_Generation_Mechanism`, then `PROFILE_CODE`, then `weight_fraction`, then `name` (a single multi-key `sort_values` call; later keys only break ties in earlier ones). It then writes the table to CSV (`FLTRD_CSV` in the configuration cell).

Earlier versions called `sort_values` four times in a row, one key each, which meant only the last call (the category) determined the order. Sorting is ascending, and rows with missing values in a key are placed last.

In [18]:
# Write keyword-filtered SPECIATE to CSV file (path set in the configuration cell)
print("Sorting keyword-filtered SPECIATE.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print("It will take a while...", "\n")

# Sort by all four keys at once (first key is the primary key, later keys break ties)
df_inchikey = df_inchikey.sort_values(
    ["CATEGORY_LEVEL_1_Generation_Mechanism", "PROFILE_CODE", "weight_fraction", "name"],
    kind="stable",
)

print("Writing keyword-filtered SPECIATE to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Save SPECIATE AFTER the InChIKey & name/keyword filtration
df_inchikey.to_csv(FLTRD_CSV, index=False, encoding="utf-8")

print(f"Keyword-filtered SPECIATE written to {FLTRD_CSV}.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")
print(f"Keyword-filtered SPECIATE CSV shape: \n{df_inchikey.shape[0]:,} rows × {df_inchikey.shape[1]} columns\n")

Sorting keyword-filtered SPECIATE. 2026-11-17 16:17:30
It will take a while... 

Writing keyword-filtered SPECIATE to CSV. 2026-11-17 16:17:31 

Keyword-filtered SPECIATE written to ./speciate_cleaned/SPECIATE_Profiles_All_02_Fltrd.csv. 2026-11-17 16:17:33 

Keyword-filtered SPECIATE CSV shape: 
248,027 rows × 34 columns



QC print of the shape and columns of the final filtered table.

In [19]:
# QC print
print(f"Keyword-filtered SPECIATE shape: \n{df_inchikey.shape[0]:,} rows × {df_inchikey.shape[1]} columns\n")

print(f"Keyword-filtered SPECIATE columns:")
for col in df_inchikey.columns:
    print(col)
print()


Keyword-filtered SPECIATE shape: 
248,027 rows × 34 columns

Keyword-filtered SPECIATE columns:
Name_Original
SPECIES_ID
weight_fraction
WEIGHT_PERCENT
UNCERTAINTY_PERCENT
SPECIES_EMISSION_RATE
PROFILE_CODE
SPECIES_NAMES_Original
INCLUDE_IN_SUM
UNCERTAINTY_METHOD
ANALYTICAL_METHOD
PHASE
SPECIES_EMISSION_RATE_UNIT
PROFILE_NAME
PROFILE_TYPE
MASTER_POLLUTANT
VERSION
CATEGORY_LEVEL_1_Generation_Mechanism
CATEGORY_LEVEL_2_Sector_Equipment
CATEGORY_LEVEL_3_ Fuel_Product
Keywords
CAS
CAS#
CAS no hyphen
DSSTox_ID
Name
Molecular Formula
Smiles Notation
SPECIES_NAMES
CID
Feature name
IUPACName
InChIKey
Lookup_method



### Step 8: Descriptive statistics of the filtered SPECIATE

The next cells compute summary statistics on the final filtered table (`df_inchikey`). This cell only creates the empty accumulators (sets, dicts, counters) that the following cells fill.

In [20]:
# Statistics accumulators
print("Creating statistics accumulators.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

df_stat = df_inchikey

total_rows = len(df_stat)

unique_profiles = set()
unique_orig_names = set()
unique_feat_names = set()

casrn_profile_stats = defaultdict(lambda: {
    "name": None,
    "row_count": 0,
    "profiles": set(),
    "weight_fraction": []
})

column_null_counts = {}
overall_values_per_column = {}
unique_values_per_column = {}

value_counts_limited = {}  # optional sample column

print("Statistics accumulators created.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


Creating statistics accumulators. 2026-11-17 16:17:34 

Statistics accumulators created. 2026-11-17 16:17:34 



Column-level statistics: for every column, the set of distinct non-null values, the number of non-null values and the number of nulls.

In [21]:
# Column-level statistics
print("Calculating column-level statistics.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

for col in df_stat.columns:
    non_null = df_stat[col].dropna()

    unique_values_per_column[col] = set(non_null.unique())
    overall_values_per_column[col] = non_null.shape[0]
    column_null_counts[col] = df_stat[col].isna().sum()

print("Column-level statistics calculated.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


Calculating column-level statistics. 2026-11-17 16:17:34 

Column-level statistics calculated. 2026-11-17 16:17:35 



Collect the distinct profile names, original compound names (`orig_name`) and PubChem-based names (`name`) in the filtered table.

In [22]:
# Global unique entities
print("Calculating global unique entities.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

unique_profiles.update(df_stat["PROFILE_NAME"].dropna().unique())
unique_orig_names.update(df_stat["orig_name"].dropna().unique())
unique_feat_names.update(df_stat["name"].dropna().unique())

print("Global unique entities calculated.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


Calculating global unique entities. 2026-11-17 16:17:35 

Global unique entities calculated. 2026-11-17 16:17:35 



Aggregate per `cas`: row count, first-seen feature name, set of profiles it occurs in, and the list of all `weight_fraction` values. Rows without a `cas` (NaN, or the text `nan`/`None`) are skipped and counted. This uses `iterrows()`, so it is slow on large tables.

In [23]:
# CAS# → profile/weight fraction relationships
print("Establishing CAS# & profile/weight fraction relationships.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

n_skipped_no_cas = 0

for _, row in df_stat.iterrows():
    casrn = row.get("cas")

    # Missing CAS# is NaN, or the text 'nan' after the astype(str) before the exports
    if pd.isna(casrn) or str(casrn).strip().lower() in ("nan", "none", ""):
        n_skipped_no_cas += 1
        continue

    s = casrn_profile_stats[casrn]

    if "weight_fraction" not in s:
        s["weight_fraction"] = []
    if "profiles" not in s:
        s["profiles"] = set()
    if "row_count" not in s:
        s["row_count"] = 0
    if "name" not in s:
        s["name"] = None

    # Row-level count
    s["row_count"] += 1

    # Chemical name (store once)
    chem_name = row.get("name")
    if pd.notna(chem_name) and s["name"] is None:
        s["name"] = chem_name

    # Profile names
    profile = row.get("PROFILE_NAME")
    if pd.notna(profile):
        s["profiles"].add(profile)

    # Weight fractions
    cwf = row.get("weight_fraction")
    if pd.notna(cwf):
        s["weight_fraction"].append(cwf)

print(f"Skipped {n_skipped_no_cas:,} rows without a CAS#.")
print(f"{len(casrn_profile_stats):,} distinct CAS# aggregated.\n")
print("CAS# & profile/weight fraction relationships established.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

Establishing CAS# & profile/weight fraction relationships. 2026-11-17 16:17:35 

Skipped 25,238 rows without a CAS#.
1,962 distinct CAS# aggregated.

CAS# & profile/weight fraction relationships established. 2026-11-17 16:17:42 



Summary table: for every column, the number of non-null values ("Overall"), the number of distinct values ("Unique") and the total number of rows.

In [24]:
# Summary
print("===== OVERALL & UNIQUE VALUE COUNTS PER COLUMN =====\n")

rows = []

for col in unique_values_per_column:
    rows.append({
        "Column": col,
        "Overall": overall_values_per_column.get(col, 0),
        "Unique": len(unique_values_per_column[col]),
        "Total rows": total_rows
    })

summary_df = pd.DataFrame(rows)

summary_df.style.format({
    "Overall": "{:,}",
    "Unique": "{:,}",
    "Total rows": "{:,}"
}).set_table_styles([
    {"selector": "th, td", "props": [("border", "1px solid black"), ("text-align", "left")]},
    {"selector": "table", "props": [("border-collapse", "collapse")]}
])


===== OVERALL & UNIQUE VALUE COUNTS PER COLUMN =====



,Column,Overall,Unique,Total rows
0,Name_Original,"248,027","2,769","248,027"
1,SPECIES_ID,"248,027","2,772","248,027"
2,weight_fraction,"242,534","167,037","248,027"
3,WEIGHT_PERCENT,"242,534","167,085","248,027"
4,UNCERTAINTY_PERCENT,"246,857","57,621","248,027"
5,SPECIES_EMISSION_RATE,"5,819","4,844","248,027"
6,PROFILE_CODE,"248,027","6,760","248,027"
7,SPECIES_NAMES_Original,"245,360","2,546","248,027"
8,INCLUDE_IN_SUM,"248,027",2,"248,027"
9,UNCERTAINTY_METHOD,"86,654",13,"248,027"


Show the `TOP_N` most frequently occurring compounds (by number of rows), with their number of distinct profiles and weight-fraction min/max/mean.

Note: the `.style.format` keys `"Central WF min/max/mean"` do not match the actual column names (`"WF min/max/mean"`), so the weight-fraction columns are not number-formatted. This is cosmetic only.

In [25]:
# Show top N most frequent compounds
TOP_N = 30

# Sort CAS# by row_count descending
sorted_casrn = sorted(
    casrn_profile_stats.items(),
    key=lambda x: x[1]["row_count"],
    reverse=True
)

# Build table rows
rows = []

for casrn, stats in sorted_casrn[:TOP_N]:
    chem_name = stats["name"] or "Unknown"
    row_count = stats["row_count"]
    unique_profiles = len(stats["profiles"])

    # Wight fraction stats
    if stats["weight_fraction"]:
        cwf_min = np.min(stats["weight_fraction"])
        cwf_max = np.max(stats["weight_fraction"])
        cwf_mean = np.mean(stats["weight_fraction"])
    else:
        cwf_min = cwf_max = cwf_mean = np.nan

    rows.append({
        "cas": casrn,
        "Chemical name": chem_name,
        "Occurrences (rows)": row_count,
        "Unique profiles": unique_profiles,
        "WF min": cwf_min,
        "WF max": cwf_max,
        "WF mean": cwf_mean
    })

# Create DataFrame
casrn_df = pd.DataFrame(rows)

# Render nicely with commas and borders
casrn_df.style.format({
    "Occurrences (rows)": "{:,}",
    "Unique profiles": "{:,}",
    "Central WF min": "{:.5g}",
    "Central WF max": "{:.5g}",
    "Central WF mean": "{:.5g}"
}).set_table_styles([
    {"selector": "th, td", "props": [("border", "1px solid black"), ("text-align", "left")]},
    {"selector": "table", "props": [("border-collapse", "collapse")]}
])


,CAS#,Chemical name,Occurrences (rows),Unique profiles,WF min,WF max,WF mean
0,7439-96-5,Manganese,"3,161",646,0.000000,0.344100,0.002378
1,7440-32-6,Titanium,"2,813",623,0.000000,0.385000,0.002799
2,7440-62-2,Vanadium,"2,601",524,0.000000,0.128210,0.001121
3,10097-32-2,Bromine atom,"2,485",497,0.000000,0.366000,0.002056
4,28652-77-9,"1,3,8-Trimethylnaphthalene","2,204",103,0.000000,0.050662,0.001573
5,7439-97-6,Mercury,"2,135",331,0.000000,1.119400,0.001708
6,29062-98-4,"1,9-Dimethylphenanthrene","1,939",108,0.000000,0.009717,0.000243
7,7439-98-7,Molybdenum,"1,923",358,0.000000,0.007800,0.000153
8,7440-67-7,Zirconium,"1,874",340,0.000000,0.024227,0.000143
9,7439-95-4,Magnesium,"1,838",444,0.000000,0.183800,0.006386


### Step 9: Export per-compound statistics

Writes one row per `cas` (feature name, number of rows, number of distinct profiles, and min/mean/max `weight_fraction`) from `casrn_profile_stats` to CSV. The output path (`STATS_CSV`) is set in the configuration cell.

In [26]:
# Write stats DF to CSV file (path set in the configuration cell)
print("Writing stats DF to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

rows = []
for casrn, s in casrn_profile_stats.items():
    rows.append({
        "cas": casrn,
        "name": s["name"] or "Unknown",
        "Row count": s["row_count"],
        "Unique profiles": len(s["profiles"]),
        "WF mean": np.mean(s["weight_fraction"]) if s["weight_fraction"] else np.nan,
        "WF min": min(s["weight_fraction"]) if s["weight_fraction"] else np.nan,
        "WF max": max(s["weight_fraction"]) if s["weight_fraction"] else np.nan,
    })

pd.DataFrame(rows).to_csv(STATS_CSV, index=False, encoding="utf-8")

print(f"Stats DF written to {STATS_CSV}.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


Writing stats DF to CSV. 2026-11-17 16:17:43 

Stats DF written to ./speciate_cleaned/SPECIATE_Profiles_All_03_CAS_Profile_Composition.csv. 2026-11-17 16:17:43 

